# 掩码语言训练示例

## Step1 导入相关包

In [ ]:
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
)

## Step2 加载数据集

In [ ]:
dataset = load_from_disk("./wiki_cn_filtered/")
dataset

In [ ]:
dataset[0]

## Step3 数据集处理

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("hfl/chinese-macbert-base")

def process_func(examples):
    return tokenizer(examples["completion"], max_length=384, trunctation=True)

In [ ]:
tokenized_dataset = dataset.map(process_func,batched=True, remove_columns=dataset.column_names)
tokenized_dataset

In [ ]:
from torch.utils.data import DataLoader

dataloder = DataLoader(tokenized_dataset, batch_size=2, collate_fn=DataCollatorForLanguageModeling(tokenizer=tokenizer,mlm=True,mlm_probability=0.15))

In [ ]:
next(iter(dataloder))

In [ ]:
?tokenizer

## Step4 创建模型

In [ ]:
model = AutoModelForMaskedLM.from_pretrained("hfl/chinese-macbert-base")

## Step5 配置训练参数

In [ ]:
args = TrainingArguments(
    output_dir="./masked_lm",
    per_device_train_batch_size=32,
    logging_steps=10,
    num_train_epochs=1,
)

## Stept6 创建Trainer

In [ ]:
trainer = Trainer(
    args= args,
    model=model,
    train_dataset=tokenized_dataset.select(range(100)),
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer,mlm=True,mlm_probability=0.15)
)


## Step7 模型训练

In [ ]:
trainer.train()

## Step8 模型推理

In [ ]:
from transformers import pipeline
fill_mask = pipeline("fill-mask", model=model, tokenizer=tokenizer,device=0)

In [ ]:
fill_mask("西安交通[MASK][MASK]是一所位于中国陕西省西安市的综合性大学，创建于1896年，是中国历史最悠久的高等学府之一。")